# Stage 00 v2 — Raw data preparation (ingestion delgada)

```
S00_raw_data_preparation_v2.ipynb
```

## Alcance

Esta notebook **no contiene logica de negocio**. Toda la logica de
ingestion, validacion, deteccion de gaps, manifiesto y escritura atomica
vive en `src/data/s00_raw_ingestion.py`. Esta notebook solo la invoca y
muestra los resultados.

- No modifica `notebooks/S00_raw_data_preparation.ipynb` (permanece intacta
  como evidencia historica).
- No modifica los archivos fuente en `data/00_source/`.
- No aplica ninguna transformacion temporal (sin `tz_localize`, sin
  `tz_convert`, sin filtrado de sesion, sin calendario) -- eso pertenece a
  S01.
- No descarta, corrige ni rellena filas invalidas: cualquier fila que
  falle una validacion detiene la generacion del artefacto.
- Los artefactos generados usan el sufijo `_v2` y no tocan los nombres
  oficiales anteriores (`mnq_raw.parquet`, `mnq_raw_summary.json`).

## 0. Configuracion del entorno

In [1]:
# Importa Path para trabajar con rutas de archivos y carpetas.
from pathlib import Path

# Importa sys para acceder a las rutas donde Python busca módulos.
import sys


def find_project_root(
    start: Path,
    marker: str = "config/data_config.yaml",
    max_levels: int = 15
) -> Path:
    """
    Busca hacia arriba la carpeta que contiene el archivo marcador.
    En este caso, el marcador es config/data_config.yaml.
    """

    # Convierte la ruta inicial en una ruta absoluta.
    current = start.resolve()

    # Revisa como máximo 15 carpetas hacia arriba.
    for _ in range(max_levels):

        # Comprueba si el archivo marcador existe en la carpeta actual.
        if (current / marker).exists():

            # Si existe, devuelve esa carpeta como raíz del proyecto.
            return current

        # Comprueba si ya se llegó a la raíz del sistema, por ejemplo C:\.
        if current.parent == current:
            break

        # Sube un nivel en la estructura de carpetas.
        current = current.parent

    # Detiene la ejecución si no encuentra la raíz del proyecto.
    raise FileNotFoundError(
        f"No se encontro un ancestro de {start} "
        f"que contenga {marker} en {max_levels} niveles"
    )


# Obtiene la carpeta actual y busca desde allí la raíz del proyecto.
PROJECT_ROOT = find_project_root(Path.cwd())

# Muestra la ruta encontrada.
print("Project root:", PROJECT_ROOT)

# Comprueba si la raíz del proyecto todavía no está en sys.path.
if str(PROJECT_ROOT) not in sys.path:

    # La agrega para poder importar módulos como src.data.
    sys.path.insert(0, str(PROJECT_ROOT))

Project root: C:\Users\heguu\OneDrive\Escritorio\neural_profit


In [2]:
# Importa el módulo de ingestión de datos crudos de S00.
# El alias "ing" permite llamarlo de forma más corta.
from src.data import s00_raw_ingestion as ing


# Construye la ruta completa al archivo de configuración de S00.
CONFIG_PATH = PROJECT_ROOT / "config" / "data_config.yaml"

# Muestra la ruta del archivo de configuración que se utilizará.
print("Config:", CONFIG_PATH)

Config: C:\Users\heguu\OneDrive\Escritorio\neural_profit\config\data_config.yaml


## 1. Ejecucion productiva

Genera (o reutiliza, si nada cambio segun la regla de staleness de
`config/data_config.yaml` + hashes de codigo/fuentes) los artefactos
oficiales de S00 v2 en `data/01_raw/`.

In [3]:
# Ejecuta todo el proceso de ingestión de S00.
# Recibe la raíz del proyecto y la ruta del archivo de configuración.
# El resultado devuelto se guarda en la variable result.
result = ing.run_s00_ingestion(
    project_root=PROJECT_ROOT,
    config_path=CONFIG_PATH
)


# Indica si se reutilizó un resultado anterior válido
# o si los datos fueron procesados nuevamente.
print("Reutilizo artefacto existente:", result.reused_existing)

# Muestra la ruta del dataset consolidado en formato Parquet.
print("Parquet:", result.parquet_path)

# Muestra el hash SHA-256 del Parquet para verificar su integridad.
print("SHA-256 Parquet:", result.parquet_sha256)

# Muestra la ruta del manifiesto con hashes, fuentes y trazabilidad.
print("Manifest:", result.manifest_path)

# Muestra la ruta del archivo con el resumen general de S00.
print("Summary:", result.summary_path)

# Muestra la ruta del archivo que registra los gaps temporales detectados.
print("Gaps:", result.gaps_path)

# Muestra la ruta del CSV con el detalle de los archivos fuente procesados.
print("Source manifest CSV (derivado):", result.source_manifest_csv_path)

Reutilizo artefacto existente: False
Parquet: C:\Users\heguu\OneDrive\Escritorio\neural_profit\data\01_raw\mnq_raw_v2.parquet
SHA-256 Parquet: 902901bccee9aee1a306a7fc78e30d85ca749600729136b09edf853fd24b040f
Manifest: C:\Users\heguu\OneDrive\Escritorio\neural_profit\data\01_raw\mnq_raw_v2_manifest.json
Summary: C:\Users\heguu\OneDrive\Escritorio\neural_profit\data\01_raw\mnq_raw_v2_summary.json
Gaps: C:\Users\heguu\OneDrive\Escritorio\neural_profit\data\01_raw\mnq_raw_v2_gaps.parquet
Source manifest CSV (derivado): C:\Users\heguu\OneDrive\Escritorio\neural_profit\manifests\s00_source_manifest.csv


## 2. Resultado del dataset consolidado

In [4]:
# Obtiene el DataFrame consolidado generado por S00.
df = result.df

# Muestra sus dimensiones: cantidad de filas y columnas.
print("Shape:", df.shape)

# Muestra los nombres de las columnas.
print("Columnas:", list(df.columns))

# Muestra el timestamp más antiguo y el más reciente del índice.
print("Rango:", df.index.min(), "->", df.index.max())

# Muestra la zona horaria del índice.
# Si devuelve None, el índice todavía es tz-naive.
print("Index tz:", df.index.tz)

# Muestra las primeras cinco filas del DataFrame.
df.head()

Shape: (2329783, 6)
Columnas: ['open', 'high', 'low', 'close', 'volume', 'contract']
Rango: 2019-12-23 03:01:00 -> 2026-07-31 20:10:00
Index tz: None


,open,high,low,close,volume,contract
datetime,,,,,,
2019-12-23 03:01:00,8718.50,8718.75,8718.50,8718.50,9,H20
2019-12-23 03:02:00,8718.25,8718.25,8718.00,8718.25,14,H20
2019-12-23 03:03:00,8718.25,8718.50,8718.00,8718.25,74,H20
2019-12-23 03:04:00,8718.25,8719.00,8718.25,8718.50,10,H20
2019-12-23 03:05:00,8718.50,8719.00,8718.50,8719.00,6,H20


In [5]:
# Muestra las últimas cinco filas del DataFrame.
df.tail()

,open,high,low,close,volume,contract
datetime,,,,,,
2026-07-31 20:06:00,28362.75,28375.00,28347.75,28360.25,2656,U26
2026-07-31 20:07:00,28359.25,28380.50,28356.25,28375.00,2590,U26
2026-07-31 20:08:00,28374.00,28378.75,28361.25,28373.50,1333,U26
2026-07-31 20:09:00,28372.50,28395.00,28369.50,28391.75,2124,U26
2026-07-31 20:10:00,28392.25,28400.50,28386.00,28392.50,1663,U26


## 3. Validaciones ejecutadas

Si esta celda se ejecuto sin excepciones, significa que sobre los archivos
fuente detectados (27 al momento de esta ejecucion; la cantidad se deriva
automaticamente de `data/00_source/`, nunca se fija de antemano) se
verificaron -- sin descartar ninguna fila -- schema, parseo, timestamps,
monotonicidad, duplicados globales, duplicados por (timestamp, contract),
nulos, infinitos, precios positivos, volumen no negativo, volumen entero,
invariantes OHLC, filas exactamente duplicadas, archivos vacios y
transiciones de contrato. Ver `src/data/s00_raw_ingestion.py` para el
detalle de cada chequeo.

In [6]:
# Muestra cuántos archivos fuente fueron detectados y procesados.
print("Archivos fuente inventariados:", len(result.source_records))

# Recorre uno por uno los registros de los archivos fuente.
for r in result.source_records:

    # Imprime información resumida de cada archivo.
    print(
        f"  {r['filename']:24s} "          # Nombre del archivo, ancho fijo de 24 caracteres.
        f"{r['instrument']}{'':1s} "       # Instrumento, por ejemplo MNQ.
        f"contract={r['contract']:4s} "    # Contrato abreviado, por ejemplo H20.
        f"contract_full={r['contract_full']:7s} "  # Contrato completo, por ejemplo MNQH20.
        f"n_rows={r['n_rows']:6d} "        # Cantidad de filas del archivo.
        f"[{r['first_timestamp']} -> {r['last_timestamp']}]"  # Rango temporal.
    )

Archivos fuente inventariados: 27
  00_mnq_03_20.Last.txt    MNQ  contract=H20  contract_full=MNQH20  n_rows= 81660 [2019-12-23T03:01:00 -> 2020-03-20T13:30:00]
  01_mnq_06_20.Last.txt    MNQ  contract=M20  contract_full=MNQM20  n_rows= 86069 [2020-03-23T03:01:00 -> 2020-06-19T13:30:00]
  02_mnq_09_20.Last.txt    MNQ  contract=U20  contract_full=MNQU20  n_rows= 87482 [2020-06-22T03:01:00 -> 2020-09-18T13:30:00]
  03_mnq_12_20.Last.txt    MNQ  contract=Z20  contract_full=MNQZ20  n_rows= 86824 [2020-09-21T03:01:00 -> 2020-12-18T03:00:00]
  04_mnq_03_21.Last.txt    MNQ  contract=H21  contract_full=MNQH21  n_rows= 84599 [2020-12-21T03:01:00 -> 2021-03-19T13:30:00]
  05_mnq_06_21.Last.txt    MNQ  contract=M21  contract_full=MNQM21  n_rows= 87090 [2021-03-22T03:01:00 -> 2021-06-18T13:30:00]
  06_mnq_09_21.Last.txt    MNQ  contract=U21  contract_full=MNQU21  n_rows= 88205 [2021-06-21T03:01:00 -> 2021-09-17T13:30:00]
  07_mnq_12_21.Last.txt    MNQ  contract=Z21  contract_full=MNQZ21  n_rows= 8

## 4. Catalogo de gaps

El detalle fila a fila vive en `mnq_raw_v2_gaps.parquet` (no en el
manifest/summary, que solo contienen agregaciones). Ningun gap se clasifica
aqui como feriado/mantenimiento/jornada de trading de forma definitiva --
eso requiere el calendario y la zona horaria confirmada que S01 todavia no
tiene.

In [7]:
# Muestra la cantidad total de gaps temporales detectados en el dataset.
print(
    "Total de gaps registrados:",
    result.manifest["gaps"]["total_gaps"]
)

# Muestra cuántos gaps existen en cada rango de duración.
print(
    "Por bucket estructural:",
    result.manifest["gaps"]["by_structural_bucket"]
)

# Muestra cuántos gaps pertenecen a cada nivel de evidencia.
print(
    "Por evidence_level:",
    result.manifest["gaps"]["by_evidence_level"]
)

Total de gaps registrados: 4764
Por bucket estructural: {'2-9min': 2135, '10-70min': 1844, '70min-100h': 785}
Por evidence_level: {'structural_only': 3979, 'unconfirmed': 543, 'provisional_pattern_match': 242}


In [8]:
import json

print("Casos extraordinarios (>= umbral configurado):")
print(json.dumps(result.extraordinary_gaps, indent=2, ensure_ascii=False, default=str))

Casos extraordinarios (>= umbral configurado):
[
  {
    "gap_type_structural": "intra_file",
    "source_file_left": "04_mnq_03_21.Last.txt",
    "source_file_right": "04_mnq_03_21.Last.txt",
    "contract_left": "H21",
    "contract_right": "H21",
    "previous_timestamp": "2020-12-31 22:00:00",
    "next_timestamp": "2021-01-03 23:01:00",
    "duration_seconds": 262860.0,
    "structural_bucket": "70min-100h",
    "recurrence": 785,
    "provisional_interpretation_utc_hypothesis": "PROVISIONAL bajo hipótesis UTC: duración compatible con un cierre de fin de semana / feriado de mercado. No confirmado.",
    "evidence_level": "provisional_pattern_match"
  },
  {
    "gap_type_structural": "intra_file",
    "source_file_left": "08_mnq_03_22.Last.txt",
    "source_file_right": "08_mnq_03_22.Last.txt",
    "contract_left": "H22",
    "contract_right": "H22",
    "previous_timestamp": "2021-12-23 22:00:00",
    "next_timestamp": "2021-12-26 23:01:00",
    "duration_seconds": 262860.0,
    

In [9]:
import pandas as pd
# Carga el catálogo completo de gaps generado por S00.
gaps_path = (
    PROJECT_ROOT
    / "data"
    / "01_raw"
    / "mnq_raw_v2_gaps.parquet"
)

gaps = pd.read_parquet(gaps_path)

# Verifica que los dos gaps extraordinarios historicos documentados en
# 02_KNOWN_ISSUES_AND_INVALIDATED_RESULTS.md SS4.1-bis (S00-05, S00-06) ya
# NO esten presentes: data/00_source se actualizo y ambos quedaron
# resueltos por datos reales (ver 01_CURRENT_DECISIONS.md SS31). Se espera
# un DataFrame vacio en ambos casos -- si alguno reaparece, esta celda debe
# revisarse antes de continuar.
target_gaps = gaps[
    (
        (gaps["contract_left"] == "M23")
        & (gaps["contract_right"] == "M23")
        & (gaps["duration_seconds"] > 100 * 3600)
    )
    |
    (
        (gaps["contract_left"] == "H25")
        & (gaps["contract_right"] == "M25")
    )
].copy()

assert target_gaps.empty, (
    "Se esperaba que los gaps historicos M23 y H25->M25 ya no existieran "
    "en los datos actuales -- revisar data/00_source antes de continuar."
)
print("OK: ni el gap interno M23 ni el gap de transicion H25->M25 estan presentes en los datos vigentes.")

target_gaps[
    [
        "gap_type_structural",
        "source_file_left",
        "source_file_right",
        "contract_left",
        "contract_right",
        "previous_timestamp",
        "next_timestamp",
        "duration_seconds",
        "structural_bucket",
        "evidence_level",
    ]
]

OK: ni el gap interno M23 ni el gap de transicion H25->M25 estan presentes en los datos vigentes.


,gap_type_structural,source_file_left,source_file_right,contract_left,contract_right,previous_timestamp,next_timestamp,duration_seconds,structural_bucket,evidence_level


## 5. Notas sobre zona horaria (sin resolver en S00)

- El indice se persiste **tz-naive**. No se aplico `tz_localize` ni
  `tz_convert`.
- `timezone_assumption` es una suposicion heredada (`UTC`), **no**
  confirmada documentalmente -- ver `timezone_evidence` en el summary.
- `timestamp_semantics` (inicio vs cierre de barra) queda explicitamente
  `unknown_not_confirmed`.
- La conversion a `America/New_York` y la aplicacion de calendario
  pertenecen a S01.

In [10]:
print(json.dumps(result.summary, indent=2, ensure_ascii=False, default=str))

{
  "name": "mnq_raw_v2",
  "pipeline_version": "s00_v2",
  "shape": [
    2329783,
    6
  ],
  "columns": [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "contract"
  ],
  "index_type": "DatetimeIndex",
  "index_tz": "tz-naive (sin zona horaria en el índice persistido)",
  "timezone_assumption": "UTC",
  "timezone_evidence": "inferred_not_confirmed",
  "timestamp_semantics": "unknown_not_confirmed",
  "bar_interval": "1_minute",
  "price_type": "Last",
  "price_type_evidence": "inferred_from_filename",
  "datetime_min": "2019-12-23T03:01:00",
  "datetime_max": "2026-07-31T20:10:00",
  "n_sources": 27,
  "gaps_summary": {
    "total_gaps": 4764,
    "by_structural_bucket": {
      "2-9min": 2135,
      "10-70min": 1844,
      "70min-100h": 785
    },
    "by_evidence_level": {
      "structural_only": 3979,
      "unconfirmed": 543,
      "provisional_pattern_match": 242
    },
    "extraordinary_cases": [
      {
        "gap_type_structural": "intra_file",
      